# Cross-sectional 12-1 momentum on a halal universe

This notebook backtests a monthly-rebalanced, long-only cross-sectional momentum strategy over
the tickers in `equities-list.csv` (rows with `Asset Type == Stock`; the ETFs in that file are
excluded since the strategy ranks individual names against each other). The CSV labels these as
halal *candidates* only — independently verify current Shariah compliance before investing.

**Strategy specification**

- Universe: the halal candidate stocks listed above, whichever of them have enough price history
  at each monthly rebalance.
- Signal: 12-1 momentum — the trailing 12-month return ending one month before the rebalance date
  (the most recent month is skipped, the classic Jegadeesh-Titman specification).
- Rebalance: monthly, at each month end.
- Entry: buy into the top 20% of the ranked universe by momentum score.
- Exit / hold: a name already held is kept as long as it stays in the top 30%; it is sold once it
  falls out of the top 30%, or unconditionally re-evaluated at every monthly rebalance.
- Weighting: equal weight across whatever names are held after each rebalance.
- Position: long only. No leverage, shorting, or cash timing overlay.

Daily (not hourly) bars are used here: the strategy only needs month-end closing prices, and
requesting ~130 tickers of hourly history from IBKR would be both unnecessary and far more likely
to trip historical-data pacing limits.

## Before running

1. Install dependencies: `pip install ib-insync pandas numpy matplotlib pandas-datareader`
2. Run IBKR Trader Workstation or IB Gateway, enable API connections, and use a market-data-enabled
   account.
3. Set the host, port, and client ID below. Typical paper ports are 7497 for TWS and 4002 for
   Gateway; live ports are commonly 7496 and 4001.
4. Run the acquisition cell once. Each ticker's daily bars are cached to
   `data/daily/{TICKER}.csv`, and a summary of what downloaded is written to
   `data/daily_download_log.csv`. Re-running skips tickers that are already cached and logged
   `ok`, so it is safe to re-run if it is interrupted partway through.
5. IBKR historical data pacing rules cap you at roughly 60 requests per 10 minutes; the
   acquisition cell sleeps between tickers to stay under that, so fetching the full universe from
   scratch takes on the order of 20-25 minutes.

In [1]:
from pathlib import Path
import csv
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from ib_insync import IB, Stock, util
except ImportError as exc:
    raise ImportError('Install ib-insync first: pip install ib-insync') from exc

EQUITIES_LIST_PATH = Path('equities-list.csv')
DATA_DIR = Path('data') / 'daily'
LOG_PATH = Path('data') / 'daily_download_log.csv'

with open(EQUITIES_LIST_PATH, newline='') as f:
    equities_rows = list(csv.DictReader(f))
UNIVERSE = [row['Ticker'] for row in equities_rows if row['Asset Type'] == 'Stock']
print(f'{len(UNIVERSE)} candidate halal stocks in the universe')

EXCHANGE = 'SMART'
CURRENCY = 'USD'
START_DATE = pd.Timestamp('2009-01-01', tz='UTC')
END_DATE = pd.Timestamp.now(tz='UTC')
BAR_SIZE = '1 day'
DURATION = '20 Y'
WHAT_TO_SHOW = 'TRADES'
USE_RTH = True

IB_HOST = '172.27.224.1'  # '127.0.0.1'
IB_PORT = 7497
IB_CLIENT_ID = 31
PACING_SLEEP_SECONDS = 10.0

# Momentum strategy parameters
LOOKBACK_MONTHS = 12
SKIP_MONTHS = 1
TOP_BUY_PCT = 0.20   # enter the top 20% of the ranked universe
TOP_HOLD_PCT = 0.30  # stay in a position until it falls out of the top 30%
MIN_UNIVERSE = 20    # minimum ranked names required before trading a given month

INVESTMENT_2026_DOLLARS = 150.0
TARGET_CPI = None  # leave None to use the latest available CPI reading from FRED

132 candidate halal stocks in the universe


## Download or load the daily bars

Daily bars (unlike intraday bars) can be requested many years back in a single IBKR historical
data call, so each ticker only needs one `reqHistoricalData` request instead of a backward
year-by-year loop. The function below still checks the on-disk cache first and only opens an IBKR
connection if at least one ticker's cache is missing.

In [2]:
def normalize_bars(bars):
    frame = util.df(bars)
    if frame is None or frame.empty:
        return pd.DataFrame()
    frame['date'] = pd.to_datetime(frame['date'], utc=True)
    frame = frame.set_index('date').sort_index()
    keep = ['open', 'high', 'low', 'close', 'volume', 'average', 'barCount']
    return frame[[column for column in keep if column in frame.columns]]

def load_cached_bars(ticker):
    frame = pd.read_csv(DATA_DIR / f'{ticker}.csv', parse_dates=['date'], index_col='date')
    frame.index = pd.DatetimeIndex(frame.index).tz_convert('UTC')
    return frame.sort_index()

def download_universe():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    util.patchAsyncio()
    ib = IB()
    ib.connect(IB_HOST, IB_PORT, clientId=IB_CLIENT_ID, readonly=True, timeout=20)

    log_rows = list(csv.DictReader(open(LOG_PATH))) if LOG_PATH.exists() else []
    done = {row['ticker'] for row in log_rows if row['status'] == 'ok'}

    for i, ticker in enumerate(UNIVERSE, 1):
        cache_path = DATA_DIR / f'{ticker}.csv'
        if ticker in done and cache_path.exists():
            continue
        status, n_bars, first_date, last_date, error = 'ok', 0, '', '', ''
        try:
            contract = Stock(ticker, EXCHANGE, CURRENCY)
            qualified = ib.qualifyContracts(contract)
            if not qualified or contract.conId == 0:
                raise RuntimeError('contract could not be qualified')
            bars = ib.reqHistoricalData(
                contract, endDateTime='', durationStr=DURATION,
                barSizeSetting=BAR_SIZE, whatToShow=WHAT_TO_SHOW,
                useRTH=USE_RTH, formatDate=2, keepUpToDate=False,
            )
            frame = normalize_bars(bars)
            if frame.empty:
                status = 'no_data'
            else:
                frame.to_csv(cache_path, index_label='date')
                n_bars = len(frame)
                first_date, last_date = str(frame.index.min().date()), str(frame.index.max().date())
        except Exception as exc:
            status, error = 'error', str(exc)
        print(f'[{i}/{len(UNIVERSE)}] {ticker}: {status} n_bars={n_bars} {first_date}..{last_date} {error}')
        log_rows = [r for r in log_rows if r['ticker'] != ticker]
        log_rows.append({'ticker': ticker, 'status': status, 'n_bars': n_bars,
                          'first_date': first_date, 'last_date': last_date, 'error': error})
        with open(LOG_PATH, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=['ticker', 'status', 'n_bars', 'first_date', 'last_date', 'error'])
            writer.writeheader()
            writer.writerows(log_rows)
        time.sleep(PACING_SLEEP_SECONDS)

    ib.disconnect()

missing = [t for t in UNIVERSE if not (DATA_DIR / f'{t}.csv').exists()]
if missing:
    print(f'{len(missing)} tickers not yet cached, downloading from IBKR...')
    download_universe()
else:
    print('All tickers already cached.')

download_log = pd.read_csv(LOG_PATH) if LOG_PATH.exists() else pd.DataFrame()
if not download_log.empty:
    display(download_log['status'].value_counts().to_frame('count'))
    failed = download_log[download_log['status'] != 'ok']
    if not failed.empty:
        print('Tickers without usable data (excluded from the backtest):')
        display(failed[['ticker', 'status', 'error']])

107 tickers not yet cached, downloading from IBKR...


API connection failed: TimeoutError()


TimeoutError: 

## Build the monthly price panel

Each cached ticker's daily closes are combined into one wide panel, then resampled to the last
regular-session close of each month. A ticker with no cached data, or with an IBKR error, is
simply absent from the panel and therefore excluded from every month's ranking — this is a
survivorship-style simplification worth keeping in mind (see caveats).

In [ ]:
daily_closes = {}
for ticker in UNIVERSE:
    cache_path = DATA_DIR / f'{ticker}.csv'
    if not cache_path.exists():
        continue
    bars = load_cached_bars(ticker)
    if bars.empty:
        continue
    daily_closes[ticker] = bars['close']

price_panel = pd.DataFrame(daily_closes).sort_index()
price_panel = price_panel.loc[price_panel.index >= START_DATE]
monthly_prices = price_panel.resample('ME').last()

print(f'{monthly_prices.shape[1]} tickers with cached data, {monthly_prices.shape[0]} months '
      f'from {monthly_prices.index.min().date()} to {monthly_prices.index.max().date()}')
monthly_prices.notna().sum().sort_values().to_frame('months_available').head(10)

## Momentum signal and portfolio construction

At the rebalance point for month *t* (i.e. the close of month *t-1*), the momentum score for a
stock is its return from month *t-1-SKIP-LOOKBACK* to month *t-1-SKIP*: a 12-month return that
ends one month before the rebalance, so the most recent month's return never enters the ranking.
Names already held are kept while they stay inside the top 30%; new entries are drawn from the
top 20%. Whatever the resulting basket is, it is equally weighted, and the whole thing is
re-evaluated every month.

In [ ]:
monthly_returns = monthly_prices.pct_change()
dates = monthly_prices.index

portfolio_returns = pd.Series(index=dates, dtype=float)
holdings_log = {}
current_holdings = set()

min_i = LOOKBACK_MONTHS + SKIP_MONTHS + 1
for i in range(min_i, len(dates)):
    hold_date = dates[i]
    end_idx = i - 1 - SKIP_MONTHS
    start_idx = end_idx - LOOKBACK_MONTHS
    score = (monthly_prices.iloc[end_idx] / monthly_prices.iloc[start_idx] - 1).dropna()

    if len(score) < MIN_UNIVERSE:
        holdings_log[hold_date] = sorted(current_holdings)
        realized = monthly_returns.loc[hold_date, list(current_holdings)].dropna() if current_holdings else pd.Series(dtype=float)
        portfolio_returns.loc[hold_date] = realized.mean() if len(realized) else np.nan
        continue

    ranked = score.sort_values(ascending=False)
    n_universe = len(ranked)
    buy_cutoff = max(1, int(np.ceil(n_universe * TOP_BUY_PCT)))
    hold_cutoff = max(buy_cutoff, int(np.ceil(n_universe * TOP_HOLD_PCT)))
    buy_zone = set(ranked.index[:buy_cutoff])
    hold_zone = set(ranked.index[:hold_cutoff])

    retained = {t for t in current_holdings if t in hold_zone}
    new_holdings = (retained | buy_zone) & set(monthly_returns.columns)

    holdings_log[hold_date] = sorted(new_holdings)
    realized = monthly_returns.loc[hold_date, list(new_holdings)].dropna()
    portfolio_returns.loc[hold_date] = realized.mean() if len(realized) else np.nan
    current_holdings = new_holdings

portfolio_returns = portfolio_returns.dropna()
turnover = pd.Series({d: len(holdings_log[d]) for d in holdings_log})
print(f'{len(portfolio_returns)} tradable months, average {turnover.mean():.1f} names held per month')

### Latest rebalance: example ranking

The most recently formed basket, in the same shape as a momentum screen — ticker, trailing 12-1
return, and rank within the eligible universe that month.

In [ ]:
last_hold_date = max(holdings_log)
last_decision_idx = list(dates).index(last_hold_date) - 1 - SKIP_MONTHS
last_score = (monthly_prices.iloc[last_decision_idx] / monthly_prices.iloc[last_decision_idx - LOOKBACK_MONTHS] - 1).dropna()
example_table = last_score.sort_values(ascending=False).to_frame('trailing_12m_ex_1m_return')
example_table['rank'] = range(1, len(example_table) + 1)
example_table['held'] = example_table.index.isin(holdings_log[last_hold_date])
example_table['trailing_12m_ex_1m_return'] = (example_table['trailing_12m_ex_1m_return'] * 100).round(1)
print(f'Ranking used to form the {last_hold_date.date()} holding month:')
display(example_table.head(20))

## General statistics

Same shape of summary as the buy-and-hold benchmark notebook, computed from the strategy's
monthly return series rather than hourly closes.

In [ ]:
def max_drawdown(wealth):
    drawdown = wealth / wealth.cummax() - 1.0
    return float(drawdown.min())

wealth = (1 + portfolio_returns).cumprod()
start_date, end_date = portfolio_returns.index[0], portfolio_returns.index[-1]
years = (end_date - start_date).days / 365.2425
total_return = float(wealth.iloc[-1] - 1.0)
cagr = float(wealth.iloc[-1] ** (1 / years) - 1.0)
annualized_volatility = float(portfolio_returns.std() * np.sqrt(12))
sharpe_rf0 = float((portfolio_returns.mean() * 12) / annualized_volatility)

stats = pd.Series({
    'observations_months': len(portfolio_returns),
    'start': start_date,
    'end': end_date,
    'avg_names_held': float(turnover.reindex(portfolio_returns.index).mean()),
    'mean_monthly_return': float(portfolio_returns.mean()),
    'median_monthly_return': float(portfolio_returns.median()),
    'total_return': total_return,
    'CAGR': cagr,
    'annualized_volatility': annualized_volatility,
    'sharpe_ratio_rf0': sharpe_rf0,
    'max_drawdown': max_drawdown(wealth),
    'final_wealth_multiple': float(wealth.iloc[-1]),
})
display(stats.to_frame('value'))

## Buy-and-hold-style dollar simulation

Same purchasing-power framing as the benchmark notebook: size a 2009-era investment so it has the
same purchasing power as $150 today (via FRED CPI), run it through the strategy's compounded
return, and compare the ending value against today's $150 in real terms. This remains a
price-only, no-cost simulation: commissions, taxes, slippage, dividends, and bid/ask spread from
monthly rebalancing are all excluded.

In [ ]:
try:
    from pandas_datareader import data as web
    cpi = web.DataReader('CPIAUCSL', 'fred', '2008-12-01', pd.Timestamp.now())['CPIAUCSL'].dropna()
    cpi_2009 = float(cpi.loc[cpi.index >= '2009-01-01'].iloc[0])
    target_cpi = float(TARGET_CPI) if TARGET_CPI is not None else float(cpi.iloc[-1])
    cpi_source = 'FRED CPIAUCSL'
except Exception as exc:
    if TARGET_CPI is None:
        raise RuntimeError('FRED CPI retrieval failed. Set TARGET_CPI to a CPIAUCSL value for the target dollar period.') from exc
    cpi_2009 = 211.143
    target_cpi = float(TARGET_CPI)
    cpi_source = 'manual target CPI; Jan 2009 CPI fallback'

initial_2009_dollars = INVESTMENT_2026_DOLLARS * cpi_2009 / target_cpi
ending_value = initial_2009_dollars * wealth.iloc[-1]
nominal_gain = ending_value - initial_2009_dollars
real_gain_vs_target_dollars = ending_value - INVESTMENT_2026_DOLLARS
real_multiple_vs_target_dollars = ending_value / INVESTMENT_2026_DOLLARS

simulation = pd.Series({
    'target_dollars': INVESTMENT_2026_DOLLARS,
    'cpi_2009': cpi_2009,
    'target_cpi': target_cpi,
    'initial_investment_in_2009_dollars': initial_2009_dollars,
    'ending_nominal_value': ending_value,
    'nominal_gain': nominal_gain,
    'total_return': total_return,
    'CAGR': cagr,
    'ending_value_in_target_dollars': ending_value,
    'real_gain_vs_target_dollars': real_gain_vs_target_dollars,
    'real_multiple_vs_target_dollars': real_multiple_vs_target_dollars,
})
print(f'CPI source: {cpi_source}')
display(simulation.to_frame('value'))

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5))
wealth.plot(ax=ax, color='#176b87', title='Growth of $1 — 12-1 cross-sectional momentum, halal universe')
ax.set_ylabel('Portfolio value')
ax.axhline(1, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

## Monthly performance

Monthly returns of the strategy's equal-weighted, top-20%-in/top-30%-out momentum basket.

In [ ]:
table = (portfolio_returns * 100).rename('ret').to_frame()
table['year'] = table.index.year
table['month'] = table.index.strftime('%b')
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_table = table.pivot(index='year', columns='month', values='ret').reindex(columns=month_order)

annual_return = portfolio_returns.groupby(portfolio_returns.index.year).apply(lambda x: (1 + x).prod() - 1) * 100
monthly_table['Year'] = annual_return

monthly_summary = pd.Series({
    'months': len(portfolio_returns),
    'positive_months': int((portfolio_returns > 0).sum()),
    'pct_positive_months': float((portfolio_returns > 0).mean()),
    'avg_monthly_return': float(portfolio_returns.mean()),
    'median_monthly_return': float(portfolio_returns.median()),
    'best_month': portfolio_returns.idxmax().strftime('%Y-%m'),
    'best_month_return': float(portfolio_returns.max()),
    'worst_month': portfolio_returns.idxmin().strftime('%Y-%m'),
    'worst_month_return': float(portfolio_returns.min()),
})

styled = (
    monthly_table.round(1)
    .style.format(precision=1, na_rep='')
    .background_gradient(cmap='RdYlGn', axis=None, vmin=-15, vmax=15, subset=month_order)
    .set_caption('Monthly return (%) by year')
)
display(styled)
display(monthly_summary.to_frame('value'))

In [ ]:
colors = np.where(portfolio_returns >= 0, '#176b87', '#c0392b')
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.bar(portfolio_returns.index, portfolio_returns.values * 100, width=20, color=colors)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('12-1 momentum monthly return (%)')
ax.set_ylabel('Return (%)')
plt.tight_layout()
plt.show()

## Caveats and next improvements

- **Look-ahead / survivorship bias in the universe**: `equities-list.csv` is today's list of halal
  candidates, projected backward. Stocks that were later added to the universe (or that would have
  been screened out earlier) are not reconstructed historically, and delisted/renamed names are not
  included at all. A rigorous backtest needs a *point-in-time* halal-screened universe, not today's
  snapshot.
- **Price-only bars**: like the benchmark notebook, this uses IBKR `TRADES` bars, which are
  split-adjusted but not dividend-adjusted. Momentum strategies with monthly turnover realize
  dividends as cash that isn't reinvested here.
- **No transaction costs**: commissions, bid/ask spread, and market impact from monthly rebalancing
  across ~20-30 names are all excluded, and momentum strategies are turnover-sensitive — real costs
  would meaningfully erode the CAGR shown above.
- **Ranking universe drift**: many tickers (e.g. recent IPOs) don't have 13 months of history in
  earlier years and are simply absent from the ranking until they do; `MIN_UNIVERSE` guards against
  ranking with too few names but doesn't correct for the universe being thin/tech-and-2020s-biased
  in its earliest years.
- **Equal weight, no risk controls**: no volatility targeting, sector caps, or single-name cap
  beyond the equal-weight top-20%/top-30% band; a single name blowing up or crashing has an outsized
  effect when the eligible universe is small.
- A serious version of this strategy would need a point-in-time halal screen, total-return prices,
  realistic transaction costs, and probably industry/sector neutralization given the literature's
  finding that industry momentum explains a meaningful share of individual-stock momentum.